<a href="https://colab.research.google.com/github/thanhuy8888/-n_Systematic-Review-AI/blob/main/experiments/colab_finetune_gpu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-tune & tối ưu screening model trên Google Colab (GPU T4)

Notebook duy nhất cho toàn bộ thí nghiệm GPU của đồ án — nhanh hơn CPU local ~16 lần.

**Trước khi chạy:** `Runtime` → `Change runtime type` → chọn **T4 GPU** → Save. Chạy cell Setup trước, sau đó chọn phần cần làm:

| Phần | Nội dung | Thời gian | Trạng thái |
|---|---|---|---|
| **1A** | distil-biobert 12ep freeze2 | ~10 phút | ✅ Đã chạy 11/06/2026 — kết quả là model chính thức (ROC-AUC 87.2%, recall 79.0%) |
| **1B** | PubMedBERT đầy đủ 12 layers | ~30 phút | ✅ Đã chạy 11/06/2026 — thua distil 6/7 chỉ số (kết luận ablation) |
| **2** | Sweep 4 cấu hình + ổn định 3 seeds | ~40–50 phút | ⏳ Bước tiếp theo |

Phần 1 giữ lại để tái lập kết quả (phụ lục báo cáo); muốn tối ưu tiếp thì sau Setup nhảy thẳng xuống **Phần 2**.

## Setup (bắt buộc, chạy đầu tiên)

In [ ]:
# Kiểm tra GPU (phải thấy Tesla T4) + lấy code + cài thư viện
!nvidia-smi -L
!git clone https://github.com/thanhuy8888/-n_Systematic-Review-AI.git srai
%cd srai
!pip install -q transformers seaborn

---
# PHẦN 1 — Hai thí nghiệm cơ bản (đã chạy, giữ để tái lập)

## 1A — distil-biobert, 12 epochs, freeze 2

Công thức thắng của vòng 1: script tự lưu checkpoint epoch có val ROC-AUC cao nhất (lần chạy 11/06 đỉnh rơi ở epoch 5) và dùng nó cho đánh giá cuối.

In [ ]:
!python experiments/baselines/finetune_pubmedbert.py \
    --model nlpie/distil-biobert --epochs 12 --lr 3e-5 \
    --batch-size 16 --max-len 256 --freeze-layers 2

In [ ]:
# Đóng gói kết quả 1A để tải về
!zip -r -q ket_qua_A_distil_12ep.zip \
    sr_core/screening_model/finetuned_pubmedbert \
    sr_core/screening_model/finetuned_meta.json \
    experiments/results/transformer_confusion_matrix.png \
    experiments/results/transformer_roc_curve.png \
    experiments/results/transformer_pr_curve.png \
    experiments/results/transformer_probability_distribution.png
from google.colab import files
files.download('ket_qua_A_distil_12ep.zip')

## 1B — PubMedBERT bản đầy đủ (12 layers, freeze 6)

Kết quả 11/06: thua bản distil ở 6/7 chỉ số — bằng chứng "model to không thắng với dataset 8K mẫu" cho chương đánh giá.

⚠️ Cell này GHI ĐÈ kết quả 1A trong thư mục — chạy cell tải về của 1A trước.

In [ ]:
!python experiments/baselines/finetune_pubmedbert.py \
    --epochs 8 --lr 2e-5 \
    --batch-size 16 --max-len 256 --freeze-layers 6

In [ ]:
# Đóng gói kết quả 1B để tải về
!zip -r -q ket_qua_B_pubmedbert_full.zip \
    sr_core/screening_model/finetuned_pubmedbert \
    sr_core/screening_model/finetuned_meta.json \
    experiments/results/transformer_confusion_matrix.png \
    experiments/results/transformer_roc_curve.png \
    experiments/results/transformer_pr_curve.png \
    experiments/results/transformer_probability_distribution.png
from google.colab import files
files.download('ket_qua_B_pubmedbert_full.zip')

---
# PHẦN 2 — Sweep cấu hình + kiểm tra ổn định (bước tiếp theo)

Tìm cấu hình tốt nhất một cách **có kỷ luật, không rò rỉ dữ liệu**:

1. **Sweep 4 cấu hình** (~30 phút) quanh vùng đã biết là tốt. Cấu hình thắng được chọn theo **validation ROC-AUC** — tuyệt đối không chọn theo test (test-set fishing = gian lận số liệu).
2. **Kiểm tra ổn định** (~12 phút): train lại config thắng với 3 seed khởi tạo (tập test giữ nguyên nhờ `--init-seed`) → số **mean ± std** thuyết phục khi bảo vệ.
3. **Đóng gói** model thắng + bảng tổng hợp để tải về deploy.

**Đã rút gọn để chạy nhanh:** từ vòng 1 ta biết model 12-layer đầy đủ thua bản distil 6-layer (thí nghiệm 1B) → bỏ config BioLinkBERT; bỏ luôn `max-len 320` và `batch 32` (tốn thời gian, lực bẩy nhỏ). Chỉ còn 4 config distil quanh điểm tốt nhất.

**Có lưu cache:** mỗi config xong được lưu vào `sweep_results/`; nếu Colab rớt giữa chừng, chạy lại cell sẽ **bỏ qua các config đã xong** (in `[cache] ... bỏ qua`) thay vì train lại từ đầu.

Config `distil_lr3e5_fz2` chính là thí nghiệm 1A vô địch vòng 1 — làm mốc chuẩn (chạy lại 1 lần để lấy số validation cho việc xếp hạng).

In [ ]:
import subprocess, json, shutil, os
import pandas as pd

CONFIGS = {
    #  tên                (model,                  epochs, lr,     batch, maxlen, freeze)
    "distil_lr3e5_fz2":  ("nlpie/distil-biobert",  12, "3e-5", 16, 256, 2),  # mốc chuẩn = 1A
    "distil_lr2e5_fz2":  ("nlpie/distil-biobert",  12, "2e-5", 16, 256, 2),
    "distil_lr5e5_fz2":  ("nlpie/distil-biobert",  12, "5e-5", 16, 256, 2),
    "distil_lr3e5_fz1":  ("nlpie/distil-biobert",  12, "3e-5", 16, 256, 1),
}

META = "sr_core/screening_model/finetuned_meta.json"
CKPT_INFO = "sr_core/screening_model/finetuned_pubmedbert_ckpt/ckpt_info.json"
MODEL_DIR = "sr_core/screening_model/finetuned_pubmedbert"
CHARTS = ["transformer_confusion_matrix.png", "transformer_roc_curve.png",
          "transformer_pr_curve.png", "transformer_probability_distribution.png"]

os.makedirs("sweep_results", exist_ok=True)
rows = []
best_val = -1.0
for name, (model, ep, lr, bs, ml, fz) in CONFIGS.items():
    print(f"\n{'='*20} {name} {'='*20}")
    dest = f"sweep_results/{name}"
    cached_meta = os.path.join(dest, "finetuned_meta.json")
    cached_info = os.path.join(dest, "ckpt_info.json")
    cached_model = os.path.join(dest, "model")
    if os.path.exists(cached_meta) and os.path.exists(cached_info):
        print(f"[cache] {name} đã có kết quả — bỏ qua train lại")
        info = json.load(open(cached_info))
        meta = json.load(open(cached_meta))
    else:
        cmd = ["python", "experiments/baselines/finetune_pubmedbert.py",
               "--model", model, "--epochs", str(ep), "--lr", lr,
               "--batch-size", str(bs), "--max-len", str(ml), "--freeze-layers", str(fz)]
        try:
            subprocess.run(cmd, check=True)
        except subprocess.CalledProcessError as e:
            print(f"!! {name} LỖI ({e}) — bỏ qua, chạy config tiếp theo")
            continue
        info = json.load(open(CKPT_INFO))
        meta = json.load(open(META))
        os.makedirs(dest, exist_ok=True)
        shutil.copy(META, dest)
        shutil.copy(CKPT_INFO, dest)
        for png in CHARTS:
            shutil.copy(f"experiments/results/{png}", dest)
        # lưu trọng số riêng cho config này để resume qua phiên vẫn lấy lại được
        shutil.rmtree(cached_model, ignore_errors=True)
        shutil.copytree(MODEL_DIR, cached_model)
    rows.append({"config": name, "best_epoch": info["epoch"],
                 "val_roc_auc": round(info["val_roc_auc"], 4),
                 "val_pr_auc": round(info["val_pr_auc"], 4),
                 "test_recall": round(meta["metrics_test"]["recall"], 4),
                 "test_f1": round(meta["metrics_test"]["f1"], 4),
                 "test_roc_auc": round(meta["metrics_test"]["roc_auc"], 4),
                 "test_pr_auc": round(meta["metrics_test"]["pr_auc"], 4),
                 "test_wss95": round(meta["wss_test"], 4)})
    if info["val_roc_auc"] > best_val:
        best_val = info["val_roc_auc"]
        shutil.rmtree("sweep_results/best_model", ignore_errors=True)
        shutil.copytree(cached_model if os.path.exists(cached_model) else MODEL_DIR,
                        "sweep_results/best_model")

df = pd.DataFrame(rows).sort_values("val_roc_auc", ascending=False).reset_index(drop=True)
df.to_csv("sweep_results/sweep_summary.csv", index=False)
WINNER = df.iloc[0]["config"]
print("\n===== BẢNG TỔNG HỢP (xếp theo VAL ROC-AUC — cột test chỉ để tham khảo) =====")
print(df.to_string(index=False))
print(f"\n>>> Config thắng (theo validation): {WINNER}")

## 2.2 — Kiểm tra ổn định: 3 seeds trên config thắng

Cùng config, cùng tập test, chỉ đổi seed khởi tạo trọng số. Mean ± std cho biết con số có vững không hay chỉ là may mắn của một seed.

In [ ]:
import numpy as np

model, ep, lr, bs, ml, fz = CONFIGS[WINNER]
stab = [json.load(open(f"sweep_results/{WINNER}/finetuned_meta.json"))]  # seed 42 đã chạy ở sweep
for s in [123, 2026]:
    print(f"\n===== init-seed {s} =====")
    subprocess.run(["python", "experiments/baselines/finetune_pubmedbert.py",
                    "--model", model, "--epochs", str(ep), "--lr", lr,
                    "--batch-size", str(bs), "--max-len", str(ml),
                    "--freeze-layers", str(fz), "--init-seed", str(s)], check=True)
    stab.append(json.load(open(META)))

print(f"\n===== ỔN ĐỊNH QUA 3 SEEDS — {WINNER} (mean ± std, %) =====")
summary = {}
for k in ["precision", "recall", "f1", "roc_auc", "pr_auc"]:
    vals = [m["metrics_test"][k] for m in stab]
    summary[k] = (np.mean(vals) * 100, np.std(vals) * 100)
    print(f"  {k:10s}: {summary[k][0]:.1f} ± {summary[k][1]:.1f}")
wss_vals = [m["wss_test"] for m in stab]
print(f"  {'wss@95':10s}: {np.mean(wss_vals)*100:.1f} ± {np.std(wss_vals)*100:.1f}")
json.dump({k: {"mean": v[0], "std": v[1]} for k, v in summary.items()},
          open("sweep_results/stability_3seeds.json", "w"), indent=2)

## 2.3 — Đóng gói model thắng để tải về

Gói gồm: trọng số model thắng (seed 42), meta + 4 biểu đồ của nó, bảng sweep đầy đủ và kết quả ổn định 3 seeds.

In [ ]:
shutil.rmtree("deploy_bundle", ignore_errors=True)
os.makedirs("deploy_bundle/sr_core/screening_model", exist_ok=True)
os.makedirs("deploy_bundle/experiments/results", exist_ok=True)
shutil.copytree("sweep_results/best_model",
                "deploy_bundle/sr_core/screening_model/finetuned_pubmedbert")
shutil.copy(f"sweep_results/{WINNER}/finetuned_meta.json",
            "deploy_bundle/sr_core/screening_model/")
for png in CHARTS:
    shutil.copy(f"sweep_results/{WINNER}/{png}", "deploy_bundle/experiments/results/")
shutil.copy("sweep_results/sweep_summary.csv", "deploy_bundle/")
shutil.copy("sweep_results/stability_3seeds.json", "deploy_bundle/")
shutil.make_archive("ket_qua_round2", "zip", "deploy_bundle")
print(f"Model thắng: {WINNER}")
from google.colab import files
files.download("ket_qua_round2.zip")

---
# Đưa kết quả về máy local

Gửi file zip cho Claude Code trên máy local để phân tích + deploy, hoặc tự làm:
1. Giải nén, chép `finetuned_pubmedbert/` + `finetuned_meta.json` đè vào `sr_core/screening_model/`.
2. Chép 4 biểu đồ vào `experiments/results/`.
3. Số Bảng 5 = metrics trong `finetuned_meta.json`; số mean ± std trong `stability_3seeds.json` dùng cho phần bàn luận độ tin cậy.

**Chỉ thay model chính thức nếu config thắng vượt mốc hiện tại** (thí nghiệm 1A: test ROC-AUC 87.2%, recall 79.0%, F1 68.3%, WSS@95 33.4%). Lưu ý GPU không tái lập từng chữ số giữa các lần chạy — quyết định dựa trên chênh lệch rõ rệt (>0.5 điểm), không phải ±0.1.